# Material Identifier -Interactive Example



#### LLM-Driven Crystal Structure Identification for DFT Simulations

This notebook walks through the **Material Identifier** framework step by step.

The framework accepts a natural-language description of a crystalline material and uses Google's Gemini LLM to identify it. Then it returns structured crystallographic data suitable as input for Density Functional Theory (DFT) simulations.

**Examples of inputs the framework handles:**
- `"silicon in the diamond cubic structure"`
- `"face-centred cubic copper"`
- `"a perovskite oxide with titanium and barium"`
- `"wurtzite gallium nitride"`

### 1. Environment Setup


#### Installing dependecies and loading API keys

Install the required packages and load the API key from the `.env` file.

> ⚠️ Make sure you have a `.env` file in the project root with your Gemini and Materials Project API keys:
> ```
> GEMINI_API_KEY=your-gemini-api-key-here
> MP_API_KEY=your-materials-project-api-key-here
> ```
> Get a free key at https://aistudio.google.com/apikey


In [1]:
# Install dependencies
!pip install google-genai python-dotenv mp-api -q

# Imports
import os
import sys
import json
import time
from dotenv import load_dotenv
from google import genai

# Fix paths — notebook runs from notebooks/ folder
notebook_dir = os.path.abspath("")
project_root = os.path.dirname(notebook_dir)

# Add project root to path so we can import our modules
sys.path.insert(0, project_root)

# Load API key from project root
load_dotenv(dotenv_path=os.path.join(project_root, ".env"))

from output_schema import MaterialStructure, AtomicPosition
from prompts import build_prompt
from identifier import call_gemini, parse_response, identify_material
from validator import validate_with_mp

# Verify key loaded
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY not found — check your .env file"
print("✅ Setup complete")
print(f"Project root: {project_root}")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ Setup complete
Project root: /Users/mariaminotaki/Desktop/materials-identifier


### 2. The Output Schema Design

#### Define what the framework produces
Before looking at the LLM call, it helps to understand what we are trying to produce.

The `MaterialStructure` dataclass defines our output format. It was designed to be **DFT-code agnostic**. This means it captures all the crystallographic information needed to generate input files for any DFT code (VASP, Quantum ESPRESSO, CP2K...) without being tied to the conventions of any single one.

Key design decisions:
- **Fractional coordinates** for atomic positions — independent of lattice conventions
- **Space group symbol + number** — both Hermann-Mauguin and integer form
- **point_group** — needed for BSE symmetry analysis
- **Confidence field** — transparency about LLM uncertainty
- **validation + electronic_properties** — Materials Project cross-validation

In [2]:
# Inspect the schema
import inspect
print(inspect.getsource(MaterialStructure))

@dataclass
class MaterialStructure:
    # --- Identity ---
    formula: str                          # e.g. "GaN"
    name: str                             # e.g. "Gallium Nitride"

    # --- Crystal symmetry ---
    crystal_system: str                   # e.g. "hexagonal"
    space_group_symbol: str               # e.g. "P6_3mc"
    space_group_number: int               # e.g. 186

    # --- Lattice parameters ---
    a: float                              # Angstroms
    b: float                              # Angstroms
    c: float                              # Angstroms
    alpha: float                          # degrees
    beta: float                           # degrees
    gamma: float                          # degrees

    # --- Atomic basis ---
    atomic_positions: list[AtomicPosition]

    # --- Metadata ---
    source: str                           # e.g. "LLM-inferred"
    confidence: str                       # "high" / "medium" / "low"

    # --- Optional fields (must c

### 3. Prompt Engineering

#### Instructing Gemini to return structured crystallographic data

The prompt is the heart of the framework. It needs to:

1. Tell the model **exactly what role to play** ->expert crystallographer
2. Specify the **output format** precisely -> valid JSON only, and not markdown
3. Define **what to do when uncertain** -> use best estimate, set confidence to low
4. Handle **ambiguity** -> pick the most common stable phase

In [3]:
# Show the prompt that will be sent to Gemini
example_description = "silicon in the diamond cubic structure"
prompt = build_prompt(example_description)
print(prompt)


You are an expert crystallographer and materials scientist.

A user will describe a crystalline material in natural language.
Your job is to identify the material and return its crystallographic
properties as a JSON object.

RULES:
- Return ONLY valid JSON. No explanation, no markdown, no code blocks.
- All lattice parameters must be in Angstroms (a, b, c) and degrees (alpha, beta, gamma).
- Atomic positions must be in fractional coordinates (between 0 and 1).
- If you are not certain about a value, use your best scientific estimate and set confidence to "low" or "medium".
- If the material is ambiguous, identify the most common/stable form at ambient conditions.

Return this exact JSON structure:
{
  "formula": "chemical formula e.g. Si",
  "name": "full material name e.g. Silicon",
  "crystal_system": "one of: cubic, tetragonal, orthorhombic, hexagonal, trigonal, monoclinic, triclinic",
  "space_group_symbol": "Hermann-Mauguin symbol e.g. Fd-3m",
  "space_group_number": 227,
  "poin

### 4. Single Material Identification

#### Step-by-step walkthrough of the full pipeline

Let's run the full pipeline on one material and inspect each step:
1. Call Gemini with the prompt
2. Receive raw JSON response
3. Parse inot a `MaterialStructure` object
4. View the structured output

In [4]:
# Step 1 — Call Gemini and see the raw response
description = "silicon in the diamond cubic structure"
raw_response = call_gemini(description)

print("Raw response from Gemini:")
print(raw_response)

Raw response from Gemini:
```json
{
  "formula": "Si",
  "name": "Silicon",
  "crystal_system": "cubic",
  "space_group_symbol": "Fd-3m",
  "space_group_number": 227,
  "point_group": "m-3m",
  "a": 5.431,
  "b": 5.431,
  "c": 5.431,
  "alpha": 90.0,
  "beta": 90.0,
  "gamma": 90.0,
  "atomic_positions": [
    {
      "element": "Si",
      "x": 0.0,
      "y": 0.0,
      "z": 0.0,
      "wyckoff_position": "8a"
    },
    {
      "element": "Si",
      "x": 0.25,
      "y": 0.25,
      "z": 0.25,
      "wyckoff_position": "8a"
    }
  ],
  "source": "LLM-inferred",
  "confidence": "high",
  "notes": null
}
```


In [5]:
# Step 2 — Parse the raw response into a MaterialStructure object
material = parse_response(raw_response)

print(f"Formula:      {material.formula}")
print(f"Name:         {material.name}")
print(f"Crystal sys:  {material.crystal_system}")
print(f"Space group:  {material.space_group_symbol} (#{material.space_group_number})")
print(f"Lattice:      a={material.a}, b={material.b}, c={material.c}")
print(f"Angles:       α={material.alpha}, β={material.beta}, γ={material.gamma}")
print(f"Confidence:   {material.confidence}")
print(f"Notes:        {material.notes}")

Formula:      Si
Name:         Silicon
Crystal sys:  cubic
Space group:  Fd-3m (#227)
Lattice:      a=5.431, b=5.431, c=5.431
Angles:       α=90.0, β=90.0, γ=90.0
Confidence:   high
Notes:        None


In [6]:
# Step 3 — View the full structured JSON output
print(material.to_json())

{
  "formula": "Si",
  "name": "Silicon",
  "crystal_system": "cubic",
  "space_group_symbol": "Fd-3m",
  "space_group_number": 227,
  "a": 5.431,
  "b": 5.431,
  "c": 5.431,
  "alpha": 90.0,
  "beta": 90.0,
  "gamma": 90.0,
  "atomic_positions": [
    {
      "element": "Si",
      "x": 0.0,
      "y": 0.0,
      "z": 0.0,
      "wyckoff_position": "8a"
    },
    {
      "element": "Si",
      "x": 0.25,
      "y": 0.25,
      "z": 0.25,
      "wyckoff_position": "8a"
    }
  ],
  "source": "LLM-inferred",
  "confidence": "high",
  "point_group": "m-3m",
  "notes": null,
  "validation": null,
  "electronic_properties": null
}


In [7]:
examples = [
    ("silicon in the diamond cubic structure",          "../examples/silicon_diamond.json"),
    ("face-centred cubic copper",                       "../examples/copper_fcc.json"),
    ("a perovskite oxide with titanium and barium",     "../examples/barium_titanate.json"),
    ("wurtzite gallium nitride",                        "../examples/gallium_nitride_wurtzite.json"),
    ("iron in the body-centred cubic structure",        "../examples/iron_bcc.json"),
]

results = []
for description, path in examples:
    material, validation = identify_material(description, output_path=path)
    results.append((material, validation))
    print("-" * 50)
    time.sleep(15)


Identifying: 'silicon in the diamond cubic structure'...
Found: Silicon (Si)
Space group: Fd-3m (#227)
Confidence: high
Validating against Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/43 [00:00<?, ?it/s]

Validation status: mismatch
Message: Large difference in: a, b, c
  a: Gemini=5.431 MP=3.849 diff=1.582 ⚠️
  b: Gemini=5.431 MP=3.849 diff=1.582 ⚠️
  c: Gemini=5.431 MP=3.849 diff=1.582 ⚠️
Saved to ../examples/silicon_diamond.json
--------------------------------------------------

Identifying: 'face-centred cubic copper'...
Found: Copper (Cu)
Space group: Fm-3m (#225)
Confidence: high
Validating against Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/8 [00:00<?, ?it/s]

Validation status: mismatch
Message: Large difference in: a, b, c
  a: Gemini=3.615 MP=2.53 diff=1.085 ⚠️
  b: Gemini=3.615 MP=2.53 diff=1.085 ⚠️
  c: Gemini=3.615 MP=2.53 diff=1.085 ⚠️
Saved to ../examples/copper_fcc.json
--------------------------------------------------

Identifying: 'a perovskite oxide with titanium and barium'...
Found: Barium Titanate (BaTiO3)
Space group: P4mm (#99)
Confidence: high
Validating against Materials Project...


...

Validation status: validated
Message: All parameters within tolerance
  a: Gemini=3.992 MP=3.983 diff=0.009 ✅
  b: Gemini=3.992 MP=4.057 diff=0.065 ✅
  c: Gemini=4.036 MP=4.057 diff=0.021 ✅
Saved to ../examples/barium_titanate.json
--------------------------------------------------

Identifying: 'wurtzite gallium nitride'...
Found: Gallium Nitride (Wurtzite) (GaN)
Space group: P6_3mc (#186)
Confidence: high
Validating against Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/44 [00:00<?, ?it/s]

Validation status: validated
Message: All parameters within tolerance
  a: Gemini=3.189 MP=3.189 diff=0.0 ✅
  b: Gemini=3.189 MP=3.189 diff=0.0 ✅
  c: Gemini=5.185 MP=5.192 diff=0.007 ✅
Saved to ../examples/gallium_nitride_wurtzite.json
--------------------------------------------------

Identifying: 'iron in the body-centred cubic structure'...
Found: Iron (alpha-phase) (Fe)
Space group: Im-3m (#229)
Confidence: high
Validating against Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/10 [00:00<?, ?it/s]

Validation status: mismatch
Message: Large difference in: c
  a: Gemini=2.866 MP=2.478 diff=0.388 ✅
  b: Gemini=2.866 MP=2.478 diff=0.388 ✅
  c: Gemini=2.866 MP=4.054 diff=1.188 ⚠️
Saved to ../examples/iron_bcc.json
--------------------------------------------------


### 5. Validation Against the Materials Project

#### Crosss-checking LLP output against computed reference data

A key limitation of using an LLM alone is that lattice parameters are
inferred from training data. To address this, we cross-validate the LLM output against the Materials Project, an open database of DFT-computed materials properties.

The validator:
1. Searches the Materials Project by chemical formula
2. Selects the most stable entry (lowest energy above hull)
3. Compares lattice parameters a, b, c against Gemini's output
4. Flags any parameter differing by more than 0.5 Å
5. Retrieves electronic properties relevant for GW and BSE calculations

In [8]:
from validator import validate_with_mp

# Run on silicon
description = "silicon in the diamond cubic structure"
raw = call_gemini(description)
material = parse_response(raw)

validation = validate_with_mp(material)

print(f"Status:      {validation['status']}")
print(f"MP ID:       {validation['mp_id']}")
print(f"MP formula:  {validation['mp_formula']}")
print(f"MP space group: {validation['mp_space_group']}")
print(f"Message:     {validation['message']}")
print()
print(f"{'Parameter':<10} {'Gemini':>10} {'MP':>10} {'Diff':>10} {'Flag':>6}")
print("-" * 50)
for param, values in validation["parameter_comparison"].items():
    flag = "⚠️" if values["diff"] > 0.5 else "✅"
    print(f"{param:<10} {values['gemini']:>10} {values['mp']:>10} {values['diff']:>10} {flag:>6}")

Retrieving SummaryDoc documents:   0%|          | 0/43 [00:00<?, ?it/s]

Status:      mismatch
MP ID:       mp-149
MP formula:  Si
MP space group: Fd-3m
Message:     Large difference in: a, b, c

Parameter      Gemini         MP       Diff   Flag
--------------------------------------------------
a               5.431      3.849      1.582     ⚠️
b               5.431      3.849      1.582     ⚠️
c               5.431      3.849      1.582     ⚠️


The validation reveals an important subtlety: some mismatches are not Gemini
errors but differences in crystallographic convention. Silicon is flagged
because Gemini returns the conventional cell (a=5.431 Å) while the Materials
Project returns the primitive cell (a=3.849 Å), both describe the same structure.

GaN and BaTiO3 validate perfectly. A future version would account for cell
conventions and add direct export to VASP and other simulation packages formats.

### 7. Pymatgen Compatibility

#### Connecting to the broader computational materials science ecosystem

The `MaterialStructure` output is fully compatible with pymatgen, the standard Python library for computational materials science. The identified structure can be directly used with any pymatgen-based workflow, including structure analysis, symmetry operations, and visualization.

Since the output converts directly to a pymatgen `Structure` object, exporting to DFT input formats is straightforward:

- **VASP POSCAR** — `Poscar(structure).write_file("POSCAR")`
- **Quantum ESPRESSO** — `PWInput(structure).write_file("input.in")`
- **CP2K** — via `pymatgen.io.cp2k`

In [10]:
from pymatgen.core import Structure, Lattice
from pymatgen.analysis.structure_analyzer import SpacegroupAnalyzer

def build_pymatgen_structure(material):
    """Convert our MaterialStructure to a pymatgen Structure object."""
    lattice = Lattice.from_parameters(
        a=material.a, b=material.b, c=material.c,
        alpha=material.alpha, beta=material.beta, gamma=material.gamma
    )
    species = [atom.element for atom in material.atomic_positions]
    coords = [[atom.x, atom.y, atom.z] for atom in material.atomic_positions]
    
    return Structure(lattice, species, coords)

structure = build_pymatgen_structure(material)
print(structure)

Full Formula (Si2)
Reduced Formula: Si
abc   :   5.431000   5.431000   5.431000
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (2)
  #  SP       a     b     c
---  ----  ----  ----  ----
  0  Si    0     0     0
  1  Si    0.25  0.25  0.25
